# DSAIT4310 Assignment: SFHH Contact Network Analysis

Attendees of the SFHH conference wore sensor badges that recorded face-to-face contacts every 20 seconds. The recordings form a temporal network $G_{data}$.

- **Nodes:** 403 attendees
- **Time Steps:** $T = 3259$ of 20 seconds each
- **Contacts:** 68,679 rows $(a, b, t)$ where nodes $a$ and $b$ meet at step $t$

Contacts are undirected so $(a, b, t)$ and $(b, a, t)$ are the same. Part A studies the static network aggregated over all steps. Part B simulates information spreading along contacts in time order.

## Environment

In [1]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

df = pd.read_excel("primary.xlsx")

## A. Aggregated Network

All $T = 3259$ time steps are collapsed into one static network $G$.

- **Nodes:** every attendee who appears in the data
- **Links:** a pair is linked if they met at least once over $[1, T]$
- **Weights:** the number of contacts between a linked pair

Questions 1 to 6 ignore weights and treat every link the same. Question 7 studies the weights.

### 1. Nodes, link density and degree deviation

What is the number of nodes $N$, the link density $p$ and the standard deviation of the degree $\sqrt{Var[D]}$?

In [ ]:
p = nx.density(G)
L = G.number_of_edges()
N = G.number_of_nodes()

degrees = np.array([d for _, d in G.degree()])
std_d = degrees.std()

print(f"N: {N}")
print(f"L: {L}")
print(f"p: {p:.4f}")
print(f"std(D): {std_d:.4f}")

### 2. Degree distribution

Plot the degree distribution. Which model fits better with respect to degree distribution: Erdős-Rényi (ER) random graphs or scale-free networks? Explain how you arrived at your conclusion.

In [ ]:
plt.ylabel("count")
plt.xlabel("degree")
sns.histplot(degrees, color="orange")

> ER random graphs have a binomial or Poisson-like degree distribution. Degrees cluster tightly around the mean in a bell shape.
>
> Scale-free (power-law) networks have many nodes with few links and a long, pronounced tail of a few nodes with very many links.
>
> The plotted distribution resembles an ER random graph. It is hard to draw a firm conclusion, but there is a clear bell shape around the mean. The distribution leans to the left (many nodes with comparatively few links), but the tail is not as pronounced as expected from a scale-free network.

### 3. Degree correlation

What is the degree correlation (assortativity) $\rho_D$? What is its physical meaning?

In [ ]:
rho_D = nx.degree_assortativity_coefficient(G)

print(f"rho_D: {rho_D:.4f}")

> Degree correlation measures the preference of nodes to attach to nodes that are similar in some measure. It ranges from $-1$ to $1$. Here the correlation is very slightly negative, which implies no preference for similar nodes to connect.

### 4. Clustering coefficient

What is the clustering coefficient $C$?

In [ ]:
C = nx.average_clustering(G)

print(f"C: {C:.4f}")

### 5. Hopcount and diameter

What is the average hopcount $E[H]$ of the shortest paths between all node pairs? What is the diameter $H_{max}$?

In [ ]:
H_max = nx.diameter(G)
E_H = nx.average_shortest_path_length(G)

print(f"H_max: {H_max}")
print(f"E[H]: {E_H:.4f}")

### 6. Small-world property

Has this network the small-world property? Justify your conclusion quantitatively (Hint: Lecture 2).

> A graph is small-world if it has a high clustering coefficient and short distances. This is quantified by the small-world coefficient
>
> $$\sigma = \frac{C / C_r}{L / L_r}$$
>
> compared with an Erdős-Rényi graph of the same average degree. The graph is small-world if $\sigma > 1$, meaning $C \gg C_r$ and $L \approx L_r$.

In [ ]:
ER = nx.erdos_renyi_graph(N, L, 1110)

C_ER = nx.average_clustering(ER)
H_ER = nx.average_shortest_path_length(ER)

sigma = (C / C_ER) / (E_H / H_ER)

print(f"C: {C:.4f}")
print(f"E[H]: {E_H:.4f}")
print(f"C_ER: {C_ER:.4f}")
print(f"sigma: {sigma:.4f}")
print(f"E[H]_ER: {H_ER:.4f}")

> The small-world coefficient does not satisfy the condition, so this graph does not have the small-world property.

### 7. Link weight distribution

The weight of a link is the total number of contacts between its two nodes within $[1, T]$. Plot the probability density function of the link weight. Choose axis scales and bins so the plot is insightful.

$$f_W(x) = \lim_{\Delta x \to 0} \frac{Pr[x < W \le x + \Delta x]}{\Delta x}$$

This is the fraction of links whose weight is within each bin $(x, x + \Delta x]$ normalized by the bin size $\Delta x$. Does $W$ follow a power-law distribution? Explain how you arrived at your conclusion.

Hint: put all metrics computed for $G$ into a table.

In [ ]:
# undirected pairs so sort each row before counting
pairs = np.sort(df[["node1", "node2"]], axis=1)
weights = pd.DataFrame(pairs).value_counts().to_numpy()
bins = np.logspace(np.log10(weights.min()), np.log10(weights.max()), 20)

# density must use linear bin widths so scales are set after
sns.histplot(weights, bins=bins, stat="density", element="step")
plt.xscale("log")
plt.yscale("log")

## B. Information spreading on a temporal network

A simplified Susceptible-Infected (SI) model on the temporal network. At $t = 0$ a single seed node $s$ is infected and all other nodes are susceptible. Whenever an infected node $i$ contacts a susceptible node $j$ at time step $t$, node $j$ becomes infected in that same step. It can infect others only from the next step onward. Infected nodes stay infected forever.

For example, if seed $s$ first contacts node $m$ at $t = 5$, then $m$ is infected at $t = 5$ even though $s$ has been infected since $t = 0$. The number of infected nodes never decreases.

Simulate the process on $G_{data}$ for $N$ iterations. Each iteration uses a different seed node $i \in [1, N]$ and runs until $T = 3259$. Record the number of infected nodes $I(t)$ at each step $t$ for each iteration.

### 8. Average infection curve

Using all $N$ iterations, plot the average number of infected nodes $E[I(t)]$ with its error bar $\sqrt{Var[I(t)]}$ as a function of time step $t$.